In [ ]:
# Kaggle job 1/2: LOSO target subjects S001-S010. Enable GPU + Internet.
import codecs, os, select, shutil, subprocess, sys, time
from pathlib import Path

BRANCH = 'feature/hada-full-mamba'
REPO_URL = 'https://github.com/CuongDM1806/tcformer-test.git'
REPO_PATH = Path('/kaggle/working/tcformer-full-mamba-s001-s010')
MNE_DATA = Path('/kaggle/working/mne_data')
RESULT_ARCHIVE = Path('/kaggle/working/full_mamba_physionet_s001_s010_4p1s_bs96_ep125_results')
TRAIN_LOG = Path('/kaggle/working/full_mamba_physionet_s001_s010_4p1s_bs96_ep125.log')

def run(command, cwd=None, env=None, stream=False, log_path=None):
    command = list(map(str, command))
    print('+', ' '.join(command), flush=True)
    if not stream:
        subprocess.run(command, cwd=str(cwd) if cwd else None, env=env, check=True)
        return
    process = subprocess.Popen(command, cwd=str(cwd) if cwd else None, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=0)
    decoder = codecs.getincrementaldecoder('utf-8')(errors='replace')
    started_at = time.monotonic()
    with open(log_path, 'w', encoding='utf-8') as log_file:
        while True:
            ready, _, _ = select.select([process.stdout], [], [], 30)
            if ready:
                chunk = os.read(process.stdout.fileno(), 4096)
                if not chunk:
                    break
                output = decoder.decode(chunk)
                print(output, end='', flush=True)
                log_file.write(output); log_file.flush()
            elif process.poll() is None:
                heartbeat = f'[Kaggle heartbeat] process is running | elapsed={(time.monotonic() - started_at) / 60:.1f}m\n'
                print(heartbeat, end='', flush=True)
                log_file.write(heartbeat); log_file.flush()
            else:
                break
        remaining = decoder.decode(b'', final=True)
        if remaining:
            print(remaining, end='', flush=True); log_file.write(remaining)
    return_code = process.wait()
    if return_code != 0:
        raise RuntimeError(f'Command failed with exit code {return_code}. Full log: {log_path}')

run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'])
if (REPO_PATH / '.git').is_dir():
    run(['git', 'remote', 'set-url', 'origin', REPO_URL], cwd=REPO_PATH)
    run(['git', 'fetch', 'origin', BRANCH], cwd=REPO_PATH)
    run(['git', 'checkout', BRANCH], cwd=REPO_PATH)
    run(['git', 'pull', '--ff-only', 'origin', BRANCH], cwd=REPO_PATH)
else:
    run(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, REPO_PATH])
run(['git', 'log', '-1', '--oneline'], cwd=REPO_PATH)
UV = shutil.which('uv') or 'uv'
run([UV, 'venv', '--clear', '--python', '3.10', '.venv'], cwd=REPO_PATH)
PYTHON = REPO_PATH / '.venv/bin/python'
run([UV, 'pip', 'install', '--python', PYTHON, 'torch==2.7.1', 'torchvision==0.22.1', '--index-url', 'https://download.pytorch.org/whl/cu126'])
run([UV, 'pip', 'install', '--python', PYTHON, '-r', 'requirements.txt'], cwd=REPO_PATH)
override = "import yaml; from pathlib import Path; p=Path('configs/hada_tcformer.yaml'); c=yaml.safe_load(p.read_text()); c['subject_ids']=list(range(1,11)); c['max_epochs_loso']=125; c['preprocessing']['physionet']['trial_duration']=4.1; c['preprocessing']['physionet']['batch_size']=48; c['preprocessing']['physionet']['accumulate_grad_batches']=2; p.write_text(yaml.safe_dump(c, sort_keys=False))"
run([PYTHON, '-c', override], cwd=REPO_PATH)
run(['nvidia-smi'])
run([PYTHON, '-c', "import torch; print('PyTorch:', torch.__version__); print('CUDA:', torch.cuda.is_available()); assert torch.cuda.is_available(), 'Kaggle GPU is not enabled'; print('GPU:', torch.cuda.get_device_name(0))"])
MNE_DATA.mkdir(parents=True, exist_ok=True)
environment = os.environ.copy()
environment.update({'PYTHONUNBUFFERED': '1', 'PYTORCH_CUDA_ALLOC_CONF': 'expandable_segments:True', 'MPLBACKEND': 'Agg', 'MNE_DATA': str(MNE_DATA), 'MNE_DATASETS_EEGBCI_PATH': str(MNE_DATA)})
print('===== KAGGLE JOB 1/2 | TARGET S001-S010 | 4.1 s | MICRO-BATCH 48 x ACCUM 2 = EFFECTIVE BS 96 | 125 EPOCHS =====', flush=True)
print('Live log:', TRAIN_LOG, flush=True)
run([PYTHON, '-u', 'train_pipeline.py', '--model', 'hada_tcformer', '--dataset', 'physionet', '--loso', '--gpu_id', '0'], cwd=REPO_PATH, env=environment, stream=True, log_path=TRAIN_LOG)
archive = shutil.make_archive(str(RESULT_ARCHIVE), 'zip', root_dir=REPO_PATH, base_dir='results')
print('Job 1 complete. Kaggle output:', archive, flush=True)


In [ ]:
# Kaggle job 2/2: LOSO target subjects S011-S020. Enable GPU + Internet.
# This cell is independent; copy it to a second Kaggle Notebook to run in parallel.
import codecs, os, select, shutil, subprocess, sys, time
from pathlib import Path

BRANCH = 'feature/hada-full-mamba'
REPO_URL = 'https://github.com/CuongDM1806/tcformer-test.git'
REPO_PATH = Path('/kaggle/working/tcformer-full-mamba-s011-s020')
MNE_DATA = Path('/kaggle/working/mne_data')
RESULT_ARCHIVE = Path('/kaggle/working/full_mamba_physionet_s011_s020_4p1s_bs96_ep125_results')
TRAIN_LOG = Path('/kaggle/working/full_mamba_physionet_s011_s020_4p1s_bs96_ep125.log')

def run(command, cwd=None, env=None, stream=False, log_path=None):
    command = list(map(str, command))
    print('+', ' '.join(command), flush=True)
    if not stream:
        subprocess.run(command, cwd=str(cwd) if cwd else None, env=env, check=True)
        return
    process = subprocess.Popen(command, cwd=str(cwd) if cwd else None, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=0)
    decoder = codecs.getincrementaldecoder('utf-8')(errors='replace')
    started_at = time.monotonic()
    with open(log_path, 'w', encoding='utf-8') as log_file:
        while True:
            ready, _, _ = select.select([process.stdout], [], [], 30)
            if ready:
                chunk = os.read(process.stdout.fileno(), 4096)
                if not chunk:
                    break
                output = decoder.decode(chunk)
                print(output, end='', flush=True)
                log_file.write(output); log_file.flush()
            elif process.poll() is None:
                heartbeat = f'[Kaggle heartbeat] process is running | elapsed={(time.monotonic() - started_at) / 60:.1f}m\n'
                print(heartbeat, end='', flush=True)
                log_file.write(heartbeat); log_file.flush()
            else:
                break
        remaining = decoder.decode(b'', final=True)
        if remaining:
            print(remaining, end='', flush=True); log_file.write(remaining)
    return_code = process.wait()
    if return_code != 0:
        raise RuntimeError(f'Command failed with exit code {return_code}. Full log: {log_path}')

run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'])
if (REPO_PATH / '.git').is_dir():
    run(['git', 'remote', 'set-url', 'origin', REPO_URL], cwd=REPO_PATH)
    run(['git', 'fetch', 'origin', BRANCH], cwd=REPO_PATH)
    run(['git', 'checkout', BRANCH], cwd=REPO_PATH)
    run(['git', 'pull', '--ff-only', 'origin', BRANCH], cwd=REPO_PATH)
else:
    run(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, REPO_PATH])
run(['git', 'log', '-1', '--oneline'], cwd=REPO_PATH)
UV = shutil.which('uv') or 'uv'
run([UV, 'venv', '--clear', '--python', '3.10', '.venv'], cwd=REPO_PATH)
PYTHON = REPO_PATH / '.venv/bin/python'
run([UV, 'pip', 'install', '--python', PYTHON, 'torch==2.7.1', 'torchvision==0.22.1', '--index-url', 'https://download.pytorch.org/whl/cu126'])
run([UV, 'pip', 'install', '--python', PYTHON, '-r', 'requirements.txt'], cwd=REPO_PATH)
override = "import yaml; from pathlib import Path; p=Path('configs/hada_tcformer.yaml'); c=yaml.safe_load(p.read_text()); c['subject_ids']=list(range(11,21)); c['max_epochs_loso']=125; c['preprocessing']['physionet']['trial_duration']=4.1; c['preprocessing']['physionet']['batch_size']=48; c['preprocessing']['physionet']['accumulate_grad_batches']=2; p.write_text(yaml.safe_dump(c, sort_keys=False))"
run([PYTHON, '-c', override], cwd=REPO_PATH)
run(['nvidia-smi'])
run([PYTHON, '-c', "import torch; print('PyTorch:', torch.__version__); print('CUDA:', torch.cuda.is_available()); assert torch.cuda.is_available(), 'Kaggle GPU is not enabled'; print('GPU:', torch.cuda.get_device_name(0))"])
MNE_DATA.mkdir(parents=True, exist_ok=True)
environment = os.environ.copy()
environment.update({'PYTHONUNBUFFERED': '1', 'PYTORCH_CUDA_ALLOC_CONF': 'expandable_segments:True', 'MPLBACKEND': 'Agg', 'MNE_DATA': str(MNE_DATA), 'MNE_DATASETS_EEGBCI_PATH': str(MNE_DATA)})
print('===== KAGGLE JOB 2/2 | TARGET S011-S020 | 4.1 s | MICRO-BATCH 48 x ACCUM 2 = EFFECTIVE BS 96 | 125 EPOCHS =====', flush=True)
print('Live log:', TRAIN_LOG, flush=True)
run([PYTHON, '-u', 'train_pipeline.py', '--model', 'hada_tcformer', '--dataset', 'physionet', '--loso', '--gpu_id', '0'], cwd=REPO_PATH, env=environment, stream=True, log_path=TRAIN_LOG)
archive = shutil.make_archive(str(RESULT_ARCHIVE), 'zip', root_dir=REPO_PATH, base_dir='results')
print('Job 2 complete. Kaggle output:', archive, flush=True)
